## ***Import Libraries***

All required libraries are imported here.

In [2]:
import pyodbc
import pandas as pd
from sqlalchemy import create_engine
import urllib
import sys
import os
sys.path.append(r'C:\Users\ujay\Desktop\CodeAlpha_DataAnalysis')

from config import SERVER, DATABASE

## ***Load Data from SQL Server***

Data is loaded directly from the `cleaned` schema in the
`ecommerce_analysis` SQL Server database using `pd.read_sql()`.


In [3]:
# connect to database
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={SERVER};'
    f'DATABASE={DATABASE};'
    f'Trusted_Connection=yes;'
)

print("Connection successful!")

Connection successful!


In [4]:
# load tabel from the schema
orders = pd.read_sql("SELECT * FROM cleaned.orders", conn)
customers = pd.read_sql("SELECT * FROM cleaned.customers", conn)
order_items = pd.read_sql("SELECT * FROM cleaned.order_items", conn)
payments = pd.read_sql("SELECT * FROM cleaned.payments", conn)
reviews = pd.read_sql("SELECT * FROM cleaned.reviews", conn)
products = pd.read_sql("SELECT * FROM cleaned.products", conn)
sellers = pd.read_sql("SELECT * FROM cleaned.sellers", conn)
geolocation = pd.read_sql("SELECT * FROM cleaned.geolocation", conn)
product_category = pd.read_sql("SELECT * FROM cleaned.category_translation", conn)

C:\Users\ujay\AppData\Local\Temp\ipykernel_8668\656148436.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders = pd.read_sql("SELECT * FROM cleaned.orders", conn)
C:\Users\ujay\AppData\Local\Temp\ipykernel_8668\656148436.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql("SELECT * FROM cleaned.customers", conn)
C:\Users\ujay\AppData\Local\Temp\ipykernel_8668\656148436.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  order_items = pd.read_sql("SELECT * FROM cleaned.order_items", conn)
C:\Use

In [5]:
# Display all column names for each table
tables = {
    'orders': orders,
    'customers': customers,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation': geolocation,
    'product_category': product_category
}

for name, df in tables.items():
    print(f"\n{'.'*50}")
    print(f"{name} :  {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")



..................................................
orders :  99441 rows x 8 columns
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

..................................................
customers :  99441 rows x 5 columns
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

..................................................
order_items :  112650 rows x 7 columns
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

..................................................
payments :  103886 rows x 5 columns
Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

..................................................
reviews :  99224 rows x 7 columns
Columns: ['review_id', 'order_id', '

# ***Data Cleaning***


### ***Fix Column Names***

Several dataframes contained column names wrapped in double quotes (e.g. `"order_id"` instead of `order_id`).
This was caused during the SQL Server import process.


### **Data Value cleaning**

Row values were wrapped in quotes (e.g., `'"value"'`), blocking numeric conversion and joins.

**Solution:** Applied a `.str.strip('"')` function to targeted columns.


In [7]:
#  list af all dataframe
dataframes = [orders, customers, order_items, payments, 
              reviews, products, sellers, geolocation, product_category]

def clean_data(dfs):
    for df in dfs:
        # Fix Column Names
        df.columns = df.columns.str.replace('"', '', regex=False)
        
        # Fix Row Data 
        cols = df.select_dtypes(include=['object']).columns
        for col in cols:
            df[col] = df[col].str.replace('"', '', regex=False)
clean_data(dataframes)

In [17]:
# orders.head()
# orders.head()
# customers.head()
# order_items.head()
# payments.head()
# reviews.head().
# products.head()
# sellers.head()
# geolocation.head()
product_category.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [23]:
import pandas as pd

def wrangle():
    # Orders: Dropped internal timestamps and update dtypes
    def wrangle_orders(df):
        cols_to_drop = ['order_approved_at','order_delivered_carrier_date']
        df = df.drop(columns=cols_to_drop, errors='ignore')
        df[['order_id', 'customer_id', 'order_status']] = df[['order_id', 'customer_id', 'order_status']].astype(str)
        date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
        df[date_cols] = df[date_cols].apply(pd.to_datetime, errors='coerce')
        return df

    # Customers: Dropped redundant unique_id and update datatypes
    def wrangle_customers(df):
        cols_to_drop = ['customer_unique_id']
        df = df.drop(columns=cols_to_drop, errors='ignore')
        # Note: only customer_id is left for casting
        df[['customer_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']] = \
            df[['customer_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']].astype(str)
        return df

    # Order Items: Dropped limit date and update dtypes
    def wrangle_order_items(df):
        cols_to_drop = ['shipping_limit_date']
        df = df.drop(columns=cols_to_drop, errors='ignore')
        df[['order_id', 'order_item_id', 'product_id', 'seller_id']] = \
            df[['order_id', 'order_item_id', 'product_id', 'seller_id']].astype(str)
        df[['price', 'freight_value']] = df[['price', 'freight_value']].apply(pd.to_numeric, errors='coerce')
        return df

    # Payments: drop a column and fix datatyoes
    def wrangle_payments(df):
        cols_to_drop = ['payment_sequential']
        df = df.drop(columns=cols_to_drop, errors='ignore')
        df[['order_id', 'payment_type']] = df[['order_id', 'payment_type']].astype(str)
        df['payment_installments'] = pd.to_numeric(df['payment_installments'], errors='coerce').fillna(0).astype(int)
        df['payment_value'] = pd.to_numeric(df['payment_value'], errors='coerce')
        return df

    # Reviews
    def wrangle_reviews(df):
        cols_to_drop = ['review_comment_title', 'review_creation_date', 'review_answer_timestamp']
        df = df.drop(columns=cols_to_drop, errors='ignore')
        df[['review_id', 'order_id', 'review_comment_message']] = \
            df[['review_id', 'order_id', 'review_comment_message']].astype(str)
    
        df['review_score'] = pd.to_numeric(df['review_score'], errors='coerce').fillna(0).astype(float)
        return df

    # Products
    def wrangle_products(df):
        cols_to_drop = ['product_name_lenght', 'product_description_lenght', 'product_weight_g', 
                        'product_length_cm', 'product_height_cm', 'product_width_cm']
        df = df.drop(columns=cols_to_drop, errors='ignore')
        df[['product_id', 'product_category_name']] = df[['product_id', 'product_category_name']].astype(str)
        df['product_photos_qty'] = pd.to_numeric(df['product_photos_qty'], errors='coerce')
        return df

    # Sellers
    def wrangle_sellers(df):
        df[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']] = \
            df[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']].astype(str)
        return df

    #Geolocation
    def wrangle_geolocation(df):
        df[['geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state']] = \
            df[['geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state']].astype(str)
        df[['geolocation_lat', 'geolocation_lng']] = \
            df[['geolocation_lat', 'geolocation_lng']].apply(pd.to_numeric, errors='coerce')
        return df

    # Product Category
    def wrangle_product_category(df):
        df[['product_category_name', 'product_category_name_english']] = \
            df[['product_category_name', 'product_category_name_english']].astype(str)
        return df

    # Execution using  existing DataFrames
    return {
        'orders': wrangle_orders(orders.copy()),
        'customers': wrangle_customers(customers.copy()),
        'order_items': wrangle_order_items(order_items.copy()),
        'payments': wrangle_payments(payments.copy()),
        'reviews': wrangle_reviews(reviews.copy()),
        'products': wrangle_products(products.copy()),
        'sellers': wrangle_sellers(sellers.copy()),
        'geolocation': wrangle_geolocation(geolocation.copy()),
        'product_category': wrangle_product_category(product_category.copy())
    }

clean_dfs = wrangle()
print("Wrangling complete. Columns dropped and datatypes handled.")

Wrangling complete. Columns dropped and datatypes handled.


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32951 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [12]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [36]:
# 1. Format the connection string
params = urllib.parse.quote_plus(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={SERVER};'
    f'DATABASE={DATABASE};'
    f'Trusted_Connection=yes;'
)

# 2. Create the SQLAlchemy engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# 3. Mapping dataframes to their SQL table names
df_map = {
    "orders": orders,
    "customers": customers,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "product_category": product_category
}

# 4. Modified upload function with schema support
def upload_cleaned_data(mapping, sql_engine):
    for table_name, df in mapping.items():
        print(f"Uploading {table_name} to 'cleaned' schema...")
        # Added schema='cleaned' to target the specific schema
        df.to_sql(
            name=table_name, 
            con=sql_engine, 
            schema='cleaned', 
            if_exists='replace', 
            index=False,
            chunksize=1000 # Helpful for industrial-sized datasets
        )
    print("Database update to 'cleaned' schema complete!")

# Run the upload
upload_cleaned_data(df_map, engine)

Uploading orders to 'cleaned' schema...


C:\ProgramData\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Uploading customers to 'cleaned' schema...
Uploading order_items to 'cleaned' schema...
Uploading payments to 'cleaned' schema...
Uploading reviews to 'cleaned' schema...
Uploading products to 'cleaned' schema...
Uploading sellers to 'cleaned' schema...
Uploading geolocation to 'cleaned' schema...
Uploading product_category to 'cleaned' schema...
Database update to 'cleaned' schema complete!
